In [2]:
# 1. Mount Google Drive của bạn
from google.colab import drive
drive.mount('/content/drive')

# 2. Cài đặt các thư viện PyTorch/Transformers
!pip install -q transformers torch pillow tqdm requests


Mounted at /content/drive


In [3]:
# 3. Tải ĐA LUỒNG toàn bộ Keyframe Zips từ server BTC (Cấu hình Single-Stream Tương thích Cloudflare)
import os, sys, glob, shutil, subprocess

ZIP_URLS = [
    "https://aic-data.ledo.io.vn/map-keyframes-aic25-b1.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L21.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L22.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L23.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L24.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L25.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_a.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_b.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_c.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_d.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_e.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L27.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L28.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L29.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L30.zip",
]

os.makedirs("/content/zips", exist_ok=True)
os.makedirs("/content/keyframes", exist_ok=True)

if not shutil.which("aria2c"):
    print("⚙️ Đang tự động cài đặt aria2c & unzip trên Colab...")
    !apt-get update -y -qq && apt-get install -y -qq aria2 wget unzip

with open("/content/urls.txt", "w") as f:
    for url in ZIP_URLS:
        f.write(f"{url}\n")

print("⚡⚡⚡ Đang tải song song 8 file ZIP (Single-stream per file, tương thích Cloudflare)...")
!aria2c -j 8 -s 1 -x 1 --allow-overwrite=true -d /content/zips -i /content/urls.txt --summary-interval=5

print("\n📦 Đang giải nén toàn bộ Keyframes vào ổ cứng NVMe SSD Colab (/content/keyframes/)...")
!unzip -q -o "/content/zips/*.zip" -d /content/keyframes/
!rm -rf /content/zips
print("✅ Đã tải & giải nén 100% keyframes vào NVMe SSD Colab hoàn tất!")


⚙️ Đang tự động cài đặt aria2c & unzip trên Colab...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../wget_1.21.2-2ubuntu1.4_amd64.deb ...
Unpacking wget (1.21.2-2ubuntu1.4) over (1.21.2-2ubuntu1.1) ...
Selecting previously unselected package libc-ares2:amd64.
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up wget (1.21.2-2ubuntu1.4) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu

In [2]:
# 4. CELL VALIDATE: Kiểm tra toàn vẹn 100% Keyframe trước khi chạy Model
import os
import io
import csv
import zipfile
import urllib.request

MAP_KEYFRAMES_URL = "https://aic-data.ledo.io.vn/map-keyframes-aic25-b1.zip"
KEYFRAMES_DIR = "/content/keyframes"

def validate_keyframes_before_model():
    print("📥 1. Đang nạp danh sách mẫu chuẩn từ BTC (map-keyframes-aic25-b1.zip)...")
    req = urllib.request.Request(MAP_KEYFRAMES_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        zip_bytes = resp.read()

    expected_metadata = []
    expected_videos = set()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        csv_files = sorted([f for f in z.namelist() if f.endswith('.csv')])
        for csv_path in csv_files:
            v_id = os.path.splitext(os.path.basename(csv_path))[0]
            expected_videos.add(v_id)
            content = z.read(csv_path).decode('utf-8')
            reader = csv.DictReader(content.splitlines())
            for i, row in enumerate(reader):
                n = int(row.get('n', i + 1))
                expected_metadata.append((v_id, f"{n:03d}.jpg"))

    total_expected = len(expected_metadata)
    print(f"✅ Chuẩn BTC: {total_expected:,} khung hình keyframe thuộc {len(expected_videos)} video.\n")

    print(f"🔍 2. Đang quét toàn bộ ổ cứng NVMe Colab tại '{KEYFRAMES_DIR}'...")
    if not os.path.exists(KEYFRAMES_DIR):
        raise FileNotFoundError(f"❌ Thư mục '{KEYFRAMES_DIR}' chưa tồn tại! Bạn hãy chạy Cell 3 để tải & giải nén.")

    video_map = {}
    for root, dirs, files in os.walk(KEYFRAMES_DIR):
        for d in dirs:
            if d.startswith("L") and "_V" in d:
                video_map[d] = os.path.join(root, d)
                video_map[d.upper()] = os.path.join(root, d)
                video_map[d.lower()] = os.path.join(root, d)

    found_videos_count = len(video_map) // 3
    print(f"📁 Thư mục video tìm thấy trên đĩa: {found_videos_count} / {len(expected_videos)} video.")

    missing_count = 0
    missing_videos = set()
    for v_id, f_name in expected_metadata:
        if v_id not in video_map:
            missing_videos.add(v_id)
            missing_count += 1
            continue
        img_p = os.path.join(video_map[v_id], f_name)
        if not os.path.exists(img_p):
            missing_count += 1

    ready_count = total_expected - missing_count
    percent = (ready_count / total_expected) * 100.0
    print("\n==================================================")
    print(f"📊 KẾT QUẢ VÀO CỬA: {ready_count:,}/{total_expected:,} ({percent:.2f}%) DỮ LIỆU ĐÃ SẴN SÀNG")
    print("==================================================")

    if missing_count > 0:
        raise RuntimeError(
            f"❌ CẢNH BÁO: Còn thiếu {missing_count:,} khung hình ({len(missing_videos)} video). "
            f"Hãy chạy lại Cell 3 để hoàn tất tải file ZIP trước khi chạy Model!"
        )
    
    print("🎉 HOÀN HẢO 100%! Dữ liệu đạt 100.00% sẵn sàng chạy Model SigLIP 2!")

if __name__ == "__main__":
    validate_keyframes_before_model()


📥 1. Đang nạp danh sách mẫu chuẩn từ BTC (map-keyframes-aic25-b1.zip)...
✅ Chuẩn BTC: 177,321 khung hình keyframe thuộc 873 video.

🔍 2. Đang quét toàn bộ ổ cứng NVMe Colab tại '/content/keyframes'...
📁 Thư mục video tìm thấy trên đĩa: 582 / 873 video.

📊 KẾT QUẢ VÀO CỬA: 177,321/177,321 (100.00%) DỮ LIỆU ĐÃ SẴN SÀNG
🎉 HOÀN HẢO 100%! Dữ liệu đạt 100.00% sẵn sàng chạy Model SigLIP 2!


In [ ]:
# 5. Trích xuất SigLIP 2 VĂT CẠN GPU 100% STEADY (NO VRAM DROPS + MEMMAP NVMe) ➔ Lưu Drive
import os
import io
import gc
import csv
import time
import json
import zipfile
import urllib.request
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel

MODEL_NAME = "google/siglip2-base-patch16-224"
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/embeddings_siglip2.npy"
TMP_MEMMAP_PATH = "/content/embeddings_siglip2_tmp.dat"
MAP_KEYFRAMES_URL = "https://aic-data.ledo.io.vn/map-keyframes-aic25-b1.zip"
KEYFRAMES_DIR = "/content/keyframes"

# 🔥 CẤU HÌNH MAX CÔNG SUẤT GPU 100% STEADY (KHÔNG KHỰNG VRAM, BỘ NHỚ GIỮ NGUYÊN 10GB VRAM):
BATCH_SIZE = 800   # Batch Size 512 mượt mà GPU VRAM
NUM_WORKERS = 0    # Đọc tiến trình chính, 100% mượt mà

def download_official_btc_metadata():
    print("📥 Đang nạp metadata chuẩn từ server BTC (map-keyframes-aic25-b1.zip)...")
    req = urllib.request.Request(MAP_KEYFRAMES_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        zip_bytes = resp.read()
    
    metadata = []
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        csv_files = sorted([f for f in z.namelist() if f.endswith('.csv')])
        for csv_path in csv_files:
            video_id = os.path.splitext(os.path.basename(csv_path))[0]
            content = z.read(csv_path).decode('utf-8')
            reader = csv.DictReader(content.splitlines())
            for i, row in enumerate(reader):
                n = int(row.get('n', i + 1))
                actual_frame_idx = int(float(row.get('frame_idx', 0)))
                metadata.append({
                    'video_id': video_id,
                    'n': n,
                    'frame_filename': f'{n:03d}.jpg',
                    'frame_idx': actual_frame_idx
                })
    return metadata

def build_video_directory_map(root_dir):
    print(f"🔍 Đang quét toàn bộ ổ cứng Colab tại '{root_dir}' để ánh xạ thư mục video...")
    video_map = {}
    for root, dirs, files in os.walk(root_dir):
        for d in dirs:
            if d.startswith("L") and "_V" in d:
                video_map[d] = os.path.join(root, d)
                video_map[d.upper()] = os.path.join(root, d)
                video_map[d.lower()] = os.path.join(root, d)
    print(f"✅ Đã định vị thành công {len(video_map) // 3} thư mục video trên ổ cứng Colab!")
    return video_map

class KeyframeDataset(Dataset):
    def __init__(self, metadata, video_map):
        self.metadata = metadata
        self.video_map = video_map

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        item = self.metadata[idx]
        v_id = item["video_id"]
        f_name = item["frame_filename"]
        
        if v_id not in self.video_map:
            raise FileNotFoundError(f"❌ Không tìm thấy thư mục video '{v_id}' trên Colab! Hãy kiểm tra lại bước 3 giải nén file zip.")
            
        img_path = os.path.join(self.video_map[v_id], f_name)
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"❌ Không tìm thấy file ảnh: '{img_path}'")
            
        img = Image.open(img_path).convert("RGB")
        return img

def custom_collate(batch):
    return batch

def run_local_nvme_extraction():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🚀 Thiết bị tính toán GPU: {device}")
    if device == "cuda":
        print(f"🔥 GPU Name: {torch.cuda.get_device_name(0)}")
        print(f"🔥 GPU VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
        torch.backends.cudnn.benchmark = True
    
    video_map = build_video_directory_map(KEYFRAMES_DIR)
    metadata = download_official_btc_metadata()
    total_count = len(metadata)
    print(f"📦 Tổng cộng: {total_count} khung hình keyframe chuẩn bị trích xuất.")
    
    test_item = metadata[0]
    if test_item['video_id'] not in video_map:
        raise FileNotFoundError(f"❌ Thư mục video {test_item['video_id']} không có trên ổ cứng! Hãy chạy lại Bước 3 để tải & giải nén zips.")
    test_path = os.path.join(video_map[test_item['video_id']], test_item['frame_filename'])
    if not os.path.exists(test_path):
        raise FileNotFoundError(f"❌ Lỗi kiểm tra ảnh đầu tiên: {test_path} không tồn tại!")
    print(f"🟢 Kiểm tra mẫu ảnh đầu tiên OK: {test_path}")

    print(f"🤗 Đang nạp mô hình Google SigLIP 2 ('{MODEL_NAME}')...")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    model.eval()

    dataset = KeyframeDataset(metadata, video_map)
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device == "cuda"),
        collate_fn=custom_collate
    )

    if os.path.exists(TMP_MEMMAP_PATH):
        os.remove(TMP_MEMMAP_PATH)
    memmap_matrix = np.memmap(TMP_MEMMAP_PATH, dtype='float32', mode='w+', shape=(total_count, 768))
    
    processed = 0
    batch_counter = 0
    start_time = time.time()
    
    print(f"⚡ Bắt đầu trích xuất 100% STEADY GPU (BATCH {BATCH_SIZE} + Continuous CUDA Memory Pool)...\n")
    
    for images in dataloader:
        with torch.inference_mode(), torch.cuda.amp.autocast(enabled=(device == "cuda"), dtype=torch.float16):
            inputs = processor(images=images, return_tensors="pt").to(device, non_blocking=True)
            outputs = model.get_image_features(**inputs)
            vecs = outputs.pooler_output if hasattr(outputs, "pooler_output") else (outputs.image_embeds if hasattr(outputs, "image_embeds") else outputs[0])
            vecs = vecs / vecs.norm(dim=-1, keepdim=True)
            feats_np = vecs.cpu().to(torch.float32).numpy()
            
        bsz = len(feats_np)
        memmap_matrix[processed : processed + bsz] = feats_np
        processed += bsz
        batch_counter += 1
        
        # Flush đĩa định kỳ (Không xóa CUDA pool VRAM để GPU không bị sụt VRAM khựng lại)
        if batch_counter % 20 == 0:
            memmap_matrix.flush()
            gc.collect()
            
        percent = (processed / total_count) * 100.0
        elapsed = time.time() - start_time
        fps = processed / elapsed if elapsed > 0 else 0
        eta_sec = (total_count - processed) / fps if fps > 0 else 0
        
        print(f"Progress: {processed}/{total_count} ({percent:.2f}%) | Speed: {fps:.1f} fps | Elapsed: {elapsed:.1f}s | ETA: {eta_sec/60:.1f} min")

    memmap_matrix.flush()
    total_elapsed = time.time() - start_time
    print(f"\n🎉 HOÀN THÀNH TRÍCH XUẤT 100% ẢNH THẬT! Matrix Shape: ({total_count}, 768) trong {total_elapsed/60:.2f} phút!")
    
    print(f"💾 Đang chuyển file kết quả ma trận chuẩn vào Google Drive tại: {DRIVE_OUTPUT_PATH}...")
    final_matrix = np.array(memmap_matrix)
    np.save(DRIVE_OUTPUT_PATH, final_matrix)
    
    del memmap_matrix
    if os.path.exists(TMP_MEMMAP_PATH):
        os.remove(TMP_MEMMAP_PATH)
        
    print(f"🎉 HOÀN THÀNH 100%! File 'embeddings_siglip2.npy' đã được lưu an toàn tại Google Drive của bạn!")

if __name__ == "__main__":
    run_local_nvme_extraction()


🚀 Thiết bị tính toán GPU: cuda
🔥 GPU Name: Tesla T4
🔥 GPU VRAM Total: 14.56 GB
🔍 Đang quét toàn bộ ổ cứng Colab tại '/content/keyframes' để ánh xạ thư mục video...
✅ Đã định vị thành công 582 thư mục video trên ổ cứng Colab!
📥 Đang nạp metadata chuẩn từ server BTC (map-keyframes-aic25-b1.zip)...
📦 Tổng cộng: 177321 khung hình keyframe chuẩn bị trích xuất.
🟢 Kiểm tra mẫu ảnh đầu tiên OK: /content/keyframes/keyframes/L21_V001/001.jpg
🤗 Đang nạp mô hình Google SigLIP 2 ('google/siglip2-base-patch16-224')...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

⚡ Bắt đầu trích xuất 100% STEADY GPU (BATCH 800 + Continuous CUDA Memory Pool)...



/tmp/ipykernel_47363/3136194468.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast(enabled=(device == "cuda"), dtype=torch.float16):


Progress: 800/177321 (0.45%) | Speed: 38.0 fps | Elapsed: 21.0s | ETA: 77.4 min
Progress: 1600/177321 (0.90%) | Speed: 41.4 fps | Elapsed: 38.7s | ETA: 70.8 min
Progress: 2400/177321 (1.35%) | Speed: 43.8 fps | Elapsed: 54.8s | ETA: 66.6 min
Progress: 3200/177321 (1.80%) | Speed: 45.3 fps | Elapsed: 70.6s | ETA: 64.0 min
Progress: 4000/177321 (2.26%) | Speed: 46.6 fps | Elapsed: 85.8s | ETA: 62.0 min
Progress: 4800/177321 (2.71%) | Speed: 47.1 fps | Elapsed: 101.9s | ETA: 61.0 min
Progress: 5600/177321 (3.16%) | Speed: 47.9 fps | Elapsed: 116.9s | ETA: 59.7 min
Progress: 6400/177321 (3.61%) | Speed: 48.3 fps | Elapsed: 132.4s | ETA: 58.9 min
Progress: 7200/177321 (4.06%) | Speed: 48.2 fps | Elapsed: 149.3s | ETA: 58.8 min
Progress: 8000/177321 (4.51%) | Speed: 48.3 fps | Elapsed: 165.5s | ETA: 58.4 min
Progress: 8800/177321 (4.96%) | Speed: 48.5 fps | Elapsed: 181.3s | ETA: 57.9 min
Progress: 9600/177321 (5.41%) | Speed: 48.6 fps | Elapsed: 197.6s | ETA: 57.5 min
Progress: 10400/177321